# 18. Die Attention schnell machenAus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle# die zwei oder drei Dateien, die das Kapitel braucht. Wer das# Repository geklont hat, findet sie schon vor, und es wird# nichts heruntergeladen.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"PLACES = [".", "code", "../code"]for name in ["ch18_fast.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Was die Attention kostet, und vier Wege drumherum.Jede Position gegen jede Position ist eine quadratische Zahl von Vergleichen. Diese Datei misst diese Kosten an Speicher und Zeit und dann, was die üblichen Abhilfen tatsächlich einsparen.

In [ ]:
import timeimport numpy as nprng = np.random.default_rng(0)

In [ ]:
def softmax(scores):    shifted = scores - scores.max(axis=-1, keepdims=True)    e = np.exp(shifted)    return e / e.sum(axis=-1, keepdims=True)

Bytes der Score-Matrix, für eine Schicht und eine Sequenz.

In [ ]:
def score_memory(length, heads=32, bytes_each=2):    return heads * length * length * bytes_each

Echte Laufzeit für einen Head bei dieser Sequenzlänge.

In [ ]:
def time_attention(length, width=64, repeats=3):    q = rng.standard_normal((length, width))    k = rng.standard_normal((length, width))    v = rng.standard_normal((length, width))    best = float("inf")    for _ in range(repeats):        start = time.perf_counter()        softmax(q @ k.T / np.sqrt(width)) @ v        best = min(best, time.perf_counter() - start)    return best

Bytes der Keys und Values, die beim Generieren bleiben.

In [ ]:
def cache_sizes(length, layers=32, heads=32, d_head=128, groups=None,                bytes_each=2):    kv_heads = heads if groups is None else groups    return 2 * layers * kv_heads * length * d_head * bytes_each

Key- und Value-Zeilen, neu berechnet, wenn nichts bleibt.

In [ ]:
def work_without_cache(steps):    return sum(range(1, steps + 1))

Eine neue Zeile pro Schritt, der Rest wird gelesen.

In [ ]:
def work_with_cache(steps):    return steps

In [ ]:
def main():    print("DIE SCORE-MATRIX, EINE SCHICHT, 32 HEADS, 16 BIT")    print("  Länge      Speicher für die Scores")    for length in [512, 2048, 8192, 32768]:        mb = score_memory(length) / 1e6        shown = f"{mb:.0f} MB" if mb < 1000 else f"{mb / 1000:.1f} GB"        print(f"  {length:<10} {shown}")    print("\nGEMESSENE ZEIT FÜR EINEN HEAD, AUF DIESEM RECHNER")    print("  Länge    Sekunden    mal länger     mal die Länge")    base = time_attention(256)    for length in [256, 512, 1024, 2048]:        taken = time_attention(length)        print(f"  {length:<8} {taken:<11.4f} {taken / base:<14.1f} "              f"{length / 256:.0f}")    print("  doppelte Länge heißt ungefähr vierfache Arbeit")    print("\n1000 TOKENS ERZEUGEN, BERECHNETE KEY- UND VALUE-ZEILEN")    without = work_without_cache(1000)    with_cache = work_with_cache(1000)    print(f"  ohne Cache       {without:_}")    print(f"  mit Cache        {with_cache:_}")    print(f"  gespart          {without / with_cache:.0f}-mal die Arbeit")    print("\nWAS DER CACHE KOSTET, 32 SCHICHTEN, 8192 TOKENS")    print("  Verfahren           Key- und Value-Heads  Cache")    for name, groups in [("Multi-Head", None), ("gruppiert, 8", 8),                         ("Multi-Query", 1)]:        size = cache_sizes(8192, groups=groups)        heads = 32 if groups is None else groups        print(f"  {name:<19} {heads:<21} {size / 1e9:.2f} GB")    print("  die Queries bleiben in allen dreien bei 32 Heads")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.